In [ ]:
import sys

sys.path.append("..")
sys.path.insert(0, "../../wildlife-datasets")
sys.path.insert(0, "../../wildlife-tools")

import pickle
import warnings
from itertools import product
from tqdm import tqdm

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from wildlife_tools.inference import TopkClassifier

from animal_re_id.config import load_paths, resolve_path
from animal_re_id.datasets import SeaTurtleID2022, combine_datasets_twe_from_paths
from utils import Data, get_info, merge_data, plot_best_scores

warnings.filterwarnings("ignore")

In [ ]:
paths = load_paths("TurtlewatchEgypt")
root_results = resolve_path(paths["root_results"])

In [ ]:
dataset_name = "TurtlewatchEgypt"
# dataset_name = "SeaTurtleID2022"

matching_parts = ["bodies", "heads", "front flippers", "rear flippers"]
img_loads = ["bbox"]

ks = [1, 5]
Bs = range(1, 101)
matching_merged = "merged"

In [ ]:
if dataset_name == "TurtlewatchEgypt":
    dataset_true = combine_datasets_twe_from_paths(paths, load_segmentation=False)
    mask1 = dataset_true.metadata["identity"] != "unknown"
    mask2 = ~dataset_true.metadata["identity"].isnull()
    dataset_true = dataset_true.get_subset(mask1 & mask2)
    dataset_true.metadata["year"] = dataset_true.metadata["path"].str[:4].astype(int)
    split_years = np.arange(2018, 2026).tolist()
    pipeline_funs_priority = ("miew", 1)
    pipeline_funs_matcher = ("miew-aliked", 1.25)
    y_max1 = 150
    y_max2 = 350
elif dataset_name == "SeaTurtleID2022":
    dataset_true = SeaTurtleID2022.from_paths(load_paths(dataset_name), False).dataset
    split_years = np.arange(2011, 2023).tolist()
    pipeline_funs_priority = ("dino", 1)
    pipeline_funs_matcher = ("dino-aliked", 1.25)
    y_max1 = 150
    y_max2 = 350
else:
    raise ValueError("Dataset now known")

pipeline_funs = (pipeline_funs_priority, pipeline_funs_matcher)
identities, n_encounters_all, n_encounters_known = get_info(dataset_true, split_years)

In [ ]:
k_max = max(ks)
pars_load_all = list(
    product(
        img_loads,
        [False, True],
        split_years,
    )
)
pars_all = [
    (img_load, B, pipeline_fun, match_opposite_sides, split_year)
    for img_load, match_opposite_sides, split_year in pars_load_all
    for pipeline_fun in pipeline_funs
    for B in Bs
]

In [ ]:
B_max = max(Bs)

def restrict_similarity_by_B(similarity_miew, similarity_miew_aliked, B):
    n_rows, n_cols = similarity_miew.shape
    B = min(B, n_cols)
    top_idx = np.argpartition(-similarity_miew, B - 1, axis=1)[:, :B]

    mask = np.ones_like(similarity_miew_aliked, dtype=bool)
    rows = np.arange(n_rows)[:, None]
    mask[rows, top_idx] = False

    similarity = similarity_miew_aliked.copy()
    similarity[mask] = -np.inf
    return similarity


def encounters_to_data(similarity, groups, labels_database, matching_part, head_multiplier):
    similarity_encounters = np.vstack([similarity[idx].max(axis=0) for idx in groups.values()])
    classifier = TopkClassifier(labels_database, k=k_max, return_all=True)
    y_preds, scores, _ = classifier(similarity_encounters)
    if matching_part == "heads":
        scores = scores * head_multiplier
    unique_encounters = list(groups.keys())
    scores_part = {x: y for x, y in zip(unique_encounters, scores)}
    preds_part = {x: y for x, y in zip(unique_encounters, y_preds)}
    return Data(scores_part, preds_part, matching_part, k_max=k_max)

name_priority, head_multiplier_priority = pipeline_funs_priority
name_matcher, head_multiplier_matcher = pipeline_funs_matcher

datas = {pars: {} for pars in pars_all}
for img_load, match_opposite_sides, split_year in tqdm(pars_load_all):
    for matching_part in matching_parts:
        # Load the saved variables for both pipelines (only at B_max)
        file_name_priority = f"{dataset_name}_{name_priority}_{B_max}_{img_load}_{matching_part}_{match_opposite_sides}_{split_year}"
        with open(f"{root_results}/{dataset_name}/{file_name_priority}.pkl", "rb") as file:
            saved_vars_priority = pickle.load(file)
        similarity_priority = saved_vars_priority["similarity"]

        file_name_matcher = f"{dataset_name}_{name_matcher}_{B_max}_{img_load}_{matching_part}_{match_opposite_sides}_{split_year}"
        with open(f"{root_results}/{dataset_name}/{file_name_matcher}.pkl", "rb") as file:
            saved_vars_matcher = pickle.load(file)
        similarity_matcher_b_max = saved_vars_matcher["similarity"]

        query = saved_vars_matcher["query"]
        database = saved_vars_matcher["database"]

        # Merge the rows of the similarity matrix by encounters
        groups = query.metadata.groupby("encounter_id").indices
        labels_database = database.metadata["identity"].to_numpy()

        # Priority ("miew") does not depend on B: compute once and reuse for every B
        data_priority = encounters_to_data(similarity_priority, groups, labels_database, matching_part, head_multiplier_priority)
        for B in Bs:
            pars = (img_load, B, pipeline_funs_priority, match_opposite_sides, split_year)
            datas[pars][matching_part] = data_priority

        # Matcher ("miew-aliked") depends on B: derive B < B_max from the B_max matrices
        for B in Bs:
            similarity = (
                similarity_matcher_b_max
                if B == B_max
                else restrict_similarity_by_B(similarity_priority, similarity_matcher_b_max, B)
            )
            data_matcher = encounters_to_data(similarity, groups, labels_database, matching_part, head_multiplier_matcher)
            pars = (img_load, B, pipeline_funs_matcher, match_opposite_sides, split_year)
            datas[pars][matching_part] = data_matcher

In [ ]:
parts_all = {}

for pars in tqdm(pars_all):
    data_selection = {m: datas[pars][m] for m in matching_parts}
    data_merged = merge_data(data_selection, k_max)
    datas[pars][matching_merged] = data_merged

In [ ]:
accuracy_all_list = []
for pars in tqdm(pars_all):
    img_load, B, (name, _), match_opposite_sides, split_year = pars
    
    for matching_part in matching_parts + [matching_merged]:
        data = datas[pars][matching_part]
        for k in ks:
            accuracy = data.compute_matches(k, identities) / n_encounters_known[split_year]
            accuracy_all_list.append({
                "img_load": img_load,
                "B": B,
                "matching_part": matching_part, 
                "name": name,
                "match_opposite_sides": match_opposite_sides, 
                "split_year": split_year,
                "k": k,
                "accuracy": accuracy
            })
accuracy_all = pd.DataFrame(accuracy_all_list)

In [ ]:
group

In [ ]:
avg = (
    accuracy_all[accuracy_all["k"] == 1]
    .groupby(["img_load", "match_opposite_sides", "matching_part", "name", "B"])["accuracy"]
    .mean()
    .unstack("B")
)

colors = ["green", "red", "blue", "orange", "purple"]
for (img_load, match_opposite_sides), group in avg.groupby(level=["img_load", "match_opposite_sides"]):    
    group = group.droplevel(["img_load", "match_opposite_sides"])    
    plt.figure(figsize=(20, 5))
    for i, matching_part in enumerate(matching_parts + ["merged"]):
        plt.subplot(1, len(matching_parts) + 1, i + 1)
        for name, row in group.loc[matching_part].iterrows():
            linestyle = "-" if "-" in name else "--"
            plt.plot(row.index, row.values, linestyle, color=colors[i], label=name)
        plt.legend()
        plt.title(matching_part)
        plt.grid(alpha=0.7)
        plt.xlabel("B")
        plt.ylim([0, 1])
        if i == 0:
            plt.ylabel("accuracy")
        plt.savefig(f"{dataset_name}_accuracy_{img_load}_{match_opposite_sides}.png", dpi=300, bbox_inches="tight")


In [ ]:
avg = (
    accuracy_all[accuracy_all["k"] == 1]
    .groupby(["img_load", "match_opposite_sides", "matching_part", "name", "B"])["accuracy"]
    .mean()
    .unstack("match_opposite_sides")
)

for img_load, group in avg.groupby(level="img_load"):
    group = group.droplevel("img_load")
    diff = (group[False] - group[True]).unstack("B")

    plt.figure(figsize=(20, 5))
    for i, matching_part in enumerate(matching_parts + ["merged"]):
        plt.subplot(1, len(matching_parts) + 1, i + 1)
        for name, row in diff.loc[matching_part].iterrows():
            linestyle = "-" if "-" in name else "--"
            plt.plot(row.index, row.values, linestyle, color=colors[i], label=name)
        
        plt.legend()
        plt.title(matching_part)
        plt.grid(alpha=0.7)
        plt.xlabel("B")
        plt.ylim([-0.05, 0.05])
        if i == 0:
            plt.ylabel("gain in accuracy for ignoring opposite sides")
        plt.savefig(f"{dataset_name}_accuracy_{img_load}_gain.png", dpi=300, bbox_inches="tight")


In [ ]:
tables = {}
df = accuracy_all[accuracy_all["B"].isin([5, 10, 50, 100])]
for matching_part in df["matching_part"].unique():
    for k in ks:
        sub = df[(df["matching_part"] == matching_part) & (df["k"] == k)]
        table = sub.pivot_table(
            index=["B", "img_load", "name", "match_opposite_sides"],
            columns="split_year",
            values="accuracy",
        ).sort_index()
        table["average"] = table.mean(axis=1)
        tables[(matching_part, k)] = table

In [ ]:
for k in ks:
    table_average = pd.DataFrame()
    for matching_part in matching_parts + [matching_merged]:
        t = tables[(matching_part, k)].copy()
        t = t.drop(split_years, axis=1).rename({"average": matching_part}, axis=1)
        table_average = pd.concat((table_average, t), axis=1)

    display(table_average)

In [ ]:
name_desired, _ = pipeline_funs[-1]
best_scores = []
for pars in pars_all:
    img_load, B, (name, _), match_opposite_sides, split_year = pars
    if img_load != "bbox" or B != max(Bs) or name != name_desired or match_opposite_sides:
        continue

    data = datas[pars]["merged"]
    best_score = data.compute_matches_by_part(1, identities)
    best_scores.extend(best_score)

plot_best_scores(best_scores, matching_parts, y_max=y_max1)
plt.savefig(f"{dataset_name}_merged.png", dpi=300, bbox_inches="tight")

In [ ]:

for name_desired, _ in pipeline_funs:
    best_scores = []
    for pars in pars_all:
        img_load, B, (name, _), match_opposite_sides, split_year = pars
        if img_load != "bbox" or B != max(Bs) or name != name_desired or match_opposite_sides:
            continue

        for matching_part in matching_parts:
            data = datas[pars][matching_part]
            best_score = data.compute_matches_by_part(1, identities)
            best_scores.extend(best_score)

    plot_best_scores(best_scores, matching_parts, y_max=y_max2)
    plt.savefig(f"{dataset_name}_{name_desired}.png", dpi=300, bbox_inches="tight")